In [ ]:
import json

def count_ids(node):
    if isinstance(node, dict):
        return (1 if "id" in node else 0) + sum(count_ids(v) for v in node.values())
    if isinstance(node, list):
        return sum(count_ids(item) for item in node)
    return 0

with open("Crop_Disease_train_qwenvl.json", "r") as f:
    data = json.load(f)
    print(count_ids(data))


145261


In [ ]:
import os
import json
import csv
import time
import random
from google import genai
from google.genai import types
from google.genai.errors import APIError
from tqdm import tqdm
from itertools import islice

INPUT_JSON = "Crop_Disease_train_qwenvl.json"
OUT_DIR = "processed_cddm_2"
os.makedirs(OUT_DIR, exist_ok=True)
CSV_FILE = os.path.join(OUT_DIR, "cddm_classifier.csv")
JSONL_FILE = os.path.join(OUT_DIR, "cddm_rag.jsonl")

BATCH_SIZE = 20
MODEL_NAME = "gemini-2.0-flash"
CSV_FIELDS = ["id", "image_path", "crop", "disease", "remedy"]

try:
    client = genai.Client()
except Exception as e:
    print(f"Error initializing Gemini client: {e}")
    client = None 


def make_batch_prompt(strands):
    strands_json = json.dumps(strands, ensure_ascii=False, indent=2)
    return f"""
You are a strict data parsing expert. Extract information ONLY from what is explicitly written.

Given a list of conversation strands about crop diseases, you must process ALL of them.
For each strand, extract the following fields:
- id: The tracking ID for the strand (integer provided in the input).
- image_path: The path to the image extracted from the <img> HTML tag in the conversation. If no image is present, set to "".
- crop: The type of crop (e.g., "Apple"). If not explicitly stated, set to "".
- disease: The disease name (e.g., "Apple scab"). If the plant is healthy, set to "healthy". If not mentioned, set to "".
- remedy: Only extract if treatment/remedy/solution is explicitly mentioned. Otherwise, keep "".

STRICT RULES:
- Never infer, guess, or hallucinate.
- Only use what’s written.
- Output a JSON array with each object containing: "id", "image_path", "crop", "disease", "remedy".
- No extra text or markdown.
Input Conversations:
{strands_json}
"""


def parse_batch_with_gemini(strands):
    if not client:
        return [{"id": s['id'], "error": "Gemini client not initialized"} for s in strands]

    prompt = make_batch_prompt(strands)
    batch_ids = [s['id'] for s in strands]

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema={
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "integer"},
                            "image_path": {"type": "string"},
                            "crop": {"type": "string"},
                            "disease": {"type": "string"},
                            "remedy": {"type": "string"},
                        },
                        "required": ["id", "image_path", "crop", "disease", "remedy"]
                    }
                }
            )
        )

        if response.text:
            return json.loads(response.text)
        else:
            print(f"⚠️ Empty response for batch {batch_ids}.")
            return [{"id": _id, "error": "Empty API response."} for _id in batch_ids]

    except APIError as e:
        print(f"🚫 API Error for batch {batch_ids}: {e}")
        return [{"id": _id, "error": f"API Error: {e}"} for _id in batch_ids]

    except json.JSONDecodeError as e:
        print(f"❌ JSON Decode Error for batch {batch_ids}: {e}")
        return [{"id": _id, "error": f"JSON Decode Error: {e}"} for _id in batch_ids]

    except Exception as e:
        print(f"💥 Unexpected error for batch {batch_ids}: {e}")
        return [{"id": _id, "error": str(e)} for _id in batch_ids]


def safe_parse_batch_with_retry(strands, max_retries=5):
    for attempt in range(max_retries):
        results = parse_batch_with_gemini(strands)
        if all("error" in r and ("API Error" in r["error"] or "RESOURCE_EXHAUSTED" in r["error"]) for r in results):
            wait_time = min(60, (2 ** attempt) + random.uniform(0, 2))
            print(f"⏳ Throttled. Waiting {wait_time:.1f}s before retry #{attempt + 1}...")
            time.sleep(wait_time)
            continue
        return results

    print(f"❗ Max retries reached for batch {[s['id'] for s in strands]}")
    return results


with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

for i, strand in enumerate(data):
    strand['id'] = i + 1

processed_ids = set()
if os.path.exists(CSV_FILE):
    with open(CSV_FILE, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        processed_ids = {int(row["id"]) for row in reader}

data = [d for d in data if d["id"] not in processed_ids]
print(f"🔁 Resuming... {len(processed_ids)} already processed, {len(data)} remaining.")

total_strands = len(data)
success_count = 0
write_header = not os.path.exists(CSV_FILE) or os.stat(CSV_FILE).st_size == 0

with open(CSV_FILE, "a", newline="", encoding="utf-8") as csvfile, \
        open(JSONL_FILE, "a", encoding="utf-8") as jsonlfile:

    csv_writer = csv.DictWriter(csvfile, fieldnames=CSV_FIELDS)
    if write_header:
        csv_writer.writeheader()

    data_iterator = iter(data)
    with tqdm(total=total_strands, desc="Processing strands with Gemini") as pbar:
        while True:
            batch = list(islice(data_iterator, BATCH_SIZE))
            if not batch:
                break

            results = safe_parse_batch_with_retry(batch)

            for r in results:
                if isinstance(r, dict) and "error" not in r:
                    csv_writer.writerow(r)
                    json.dump(r, jsonlfile, ensure_ascii=False)
                    jsonlfile.write("\n")
                    success_count += 1

            csvfile.flush()
            jsonlfile.flush()
            time.sleep(1.5)  
            pbar.update(len(batch))

print("\n--- ✅ Summary ---")
print(f"Total processed this run: {success_count}")
print(f"CSV saved to {CSV_FILE}")
print(f"JSONL saved to {JSONL_FILE}")

In [1]:
import pandas as pd
df = pd.read_csv("sdsd.csv")
df.head()

,id,image_path,crop,disease,remedy
0,1,"dataset/images/Apple,Alternaria Blotch/plant_7...",Apple,Alternaria Blotch,NaN
1,2,"dataset/images/Corn,Healthy/plant_138220.jpg",Corn,healthy,NaN
2,3,"dataset/images/Tomato,Spider Mites/plant_50137...",Tomato,Spider Mites,NaN
3,4,"dataset/images/Tomato,Late Blight/plant_30879.jpg",Tomato,Late Blight,NaN
4,5,"dataset/images/Tomato,Septoria Leaf Spot/plant...",Tomato,Septoria Leaf Spot,NaN


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2399 entries, 0 to 2398
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          2399 non-null   int64 
 1   image_path  2399 non-null   object
 2   crop        2399 non-null   object
 3   disease     2399 non-null   object
 4   remedy      73 non-null     object
dtypes: int64(1), object(4)
memory usage: 93.8+ KB


In [5]:
df.to_csv("dataset/cddm_classifier.csv", index=False)

In [3]:
len(df['disease'].unique())

49